# FuelIQ — Exploratory Data Analysis
## Fuel Consumption Ratings 2023 Dataset

Analyzing vehicle fuel consumption ratings to understand:
- Key factors affecting fuel consumption
- Distribution patterns across vehicle types
- Vehicles relevant to Kenyan fleet operations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load dataset
df = pd.read_csv('../data/Fuel Consumption Ratings 2023.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nColumns:')
for col in df.columns:
    print(f'  {col}: {df[col].dtype}')
df.head()

In [ ]:
# Data overview
print('=== Dataset Info ===')
print(f'Rows: {len(df)}')
print(f'Missing values per column:')
print(df.isnull().sum())
print(f'\n=== Descriptive Statistics ===')
df.describe()

In [ ]:
# Rename columns for convenience
df.columns = ['year', 'make', 'model', 'vehicle_class', 'engine_size_l', 'cylinders',
              'transmission', 'fuel_type', 'city_l100km', 'hwy_l100km', 'comb_l100km',
              'comb_mpg', 'co2_gkm', 'co2_rating', 'smog_rating']

print('Fuel type distribution:')
print(df['fuel_type'].value_counts())
print(f'\nVehicle class distribution:')
print(df['vehicle_class'].value_counts())

In [ ]:
# Distribution of combined fuel consumption
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df['comb_l100km'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Combined Fuel Consumption (L/100km)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Fuel Consumption')
axes[0].axvline(df['comb_l100km'].mean(), color='red', linestyle='--', label=f'Mean: {df["comb_l100km"].mean():.1f}')
axes[0].legend()

axes[1].hist(df['engine_size_l'], bins=20, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Engine Size (L)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Engine Sizes')

axes[2].hist(df['co2_gkm'], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[2].set_xlabel('CO2 Emissions (g/km)')
axes[2].set_ylabel('Count')
axes[2].set_title('Distribution of CO2 Emissions')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['engine_size_l', 'cylinders', 'city_l100km', 'hwy_l100km', 'comb_l100km', 'co2_gkm', 'co2_rating', 'smog_rating']
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdYlBu_r', center=0, fmt='.2f', square=True)
plt.title('Correlation Matrix: Vehicle Specs vs Fuel Consumption')
plt.tight_layout()
plt.show()

print('Key correlations with combined fuel consumption:')
print(corr['comb_l100km'].sort_values(ascending=False))

In [ ]:
# Fuel consumption by vehicle class
class_fuel = df.groupby('vehicle_class')['comb_l100km'].mean().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
class_fuel.plot(kind='barh', color=sns.color_palette('viridis', len(class_fuel)))
plt.xlabel('Average Combined Fuel Consumption (L/100km)')
plt.title('Average Fuel Consumption by Vehicle Class')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots: fuel consumption by fuel type and cylinders
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=df, x='fuel_type', y='comb_l100km', ax=axes[0])
axes[0].set_title('Fuel Consumption by Fuel Type')
axes[0].set_xlabel('Fuel Type')
axes[0].set_ylabel('Combined (L/100km)')

sns.boxplot(data=df, x='cylinders', y='comb_l100km', ax=axes[1])
axes[1].set_title('Fuel Consumption by Cylinder Count')
axes[1].set_xlabel('Cylinders')
axes[1].set_ylabel('Combined (L/100km)')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: Engine size vs fuel consumption
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(df['engine_size_l'], df['comb_l100km'], alpha=0.5, s=20)
z = np.polyfit(df['engine_size_l'], df['comb_l100km'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['engine_size_l'].min(), df['engine_size_l'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r-', linewidth=2)
axes[0].set_xlabel('Engine Size (L)')
axes[0].set_ylabel('Combined Fuel Consumption (L/100km)')
axes[0].set_title(f'Engine Size vs Fuel Consumption (r={df["engine_size_l"].corr(df["comb_l100km"]):.3f})')

axes[1].scatter(df['co2_gkm'], df['comb_l100km'], alpha=0.5, s=20, color='green')
axes[1].set_xlabel('CO2 Emissions (g/km)')
axes[1].set_ylabel('Combined Fuel Consumption (L/100km)')
axes[1].set_title(f'CO2 vs Fuel Consumption (r={df["co2_gkm"].corr(df["comb_l100km"]):.3f})')

plt.tight_layout()
plt.show()

In [ ]:
# Kenyan fleet analysis — common makes
kenyan_makes = ['Toyota', 'Isuzu', 'Mitsubishi', 'Nissan', 'Honda', 'Suzuki', 'Subaru']
kenyan_df = df[df['make'].isin(kenyan_makes)]

print(f'Vehicles from common Kenyan fleet makes: {len(kenyan_df)} / {len(df)} ({len(kenyan_df)/len(df)*100:.1f}%)')
print(f'\nBreakdown by make:')
print(kenyan_df['make'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

make_fuel = kenyan_df.groupby('make')['comb_l100km'].mean().sort_values()
make_fuel.plot(kind='barh', ax=axes[0], color=sns.color_palette('Set2', len(make_fuel)))
axes[0].set_xlabel('Avg Combined Fuel Consumption (L/100km)')
axes[0].set_title('Kenyan Fleet Makes: Average Fuel Consumption')

sns.boxplot(data=kenyan_df, x='make', y='comb_l100km', ax=axes[1])
axes[1].set_title('Fuel Consumption Distribution by Make')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics and key insights
print('='*60)
print('KEY INSIGHTS FROM EDA')
print('='*60)
print(f'\n1. Dataset contains {len(df)} vehicles from {df["make"].nunique()} makes')
print(f'2. Engine sizes range from {df["engine_size_l"].min()}L to {df["engine_size_l"].max()}L')
print(f'3. Average combined fuel consumption: {df["comb_l100km"].mean():.1f} L/100km')
print(f'4. Strong correlation: engine_size vs fuel_consumption (r={df["engine_size_l"].corr(df["comb_l100km"]):.3f})')
print(f'5. Strong correlation: cylinders vs fuel_consumption (r={df["cylinders"].corr(df["comb_l100km"]):.3f})')
print(f'6. CO2 and fuel consumption are nearly perfectly correlated (r={df["co2_gkm"].corr(df["comb_l100km"]):.3f})')
print(f'7. Kenyan fleet makes represent {len(kenyan_df)/len(df)*100:.1f}% of dataset')
print(f'\nKey vehicle features for ML model:')
print('  - engine_size_l (strongest predictor)')
print('  - cylinders')
print('  - fuel_type')
print('  - vehicle_class')
print(f'\nBaseline consumption rates for synthetic data generation:')
print(f'  - 4-cyl diesel: ~{df[(df["cylinders"]==4)]["comb_l100km"].mean():.1f} L/100km')
print(f'  - 6-cyl diesel: ~{df[(df["cylinders"]==6)]["comb_l100km"].mean():.1f} L/100km')
print(f'  - Overall mean: {df["comb_l100km"].mean():.1f} L/100km')